# Phase 1 – Data Cleaning & Quality Check

Name: Anusha VK

Tools Used: Python, Pandas, Numpy

Objective

The objective of this phase is to assess the quality and reliability of the NexaTel dataset before proceeding with further analysis. This phase involves identifying and handling missing values, duplicate records, incorrect data types, invalid values, outliers, and referential integrity issues to ensure the dataset is accurate, consistent, and ready for analysis.

In [66]:
import pandas as pd
import numpy as np

folder = ""

file_names = [
    "customers", "subscriptions", "plans", "plan_history", "contracts", "devices",
    "billing", "payments", "recharges", "usage_voice", "usage_sms", "usage_data",
    "network_quality", "support_tickets", "complaints", "customer_feedback",
    "retention_campaigns", "marketing_campaigns", "employees", "stores",
    "regions", "cities", "states", "data_quality_issue_log",
]

dfs = {}
for name in file_names:
    dfs[name] = pd.read_csv(folder + name + ".csv")

print("Tables loaded:", len(dfs))

Tables loaded: 24


## Data Profiling

All 24 NexaTel CSV files were loaded successfully into Python. 
For each table, the number of rows, number of columns, and data types were checked.

In [67]:
#Profiling each table(checking every table, how many rows and columns it has and what type each column is stored as)
for name in dfs:
    df = dfs[name]
    print("=" * 50)
    print(name, "| rows:", df.shape[0], "| columns:", df.shape[1])
    print(df.dtypes)

customers | rows: 19076 | columns: 25
customer_id                str
first_name                 str
last_name                  str
gender                     str
date_of_birth              str
age                      int64
email                      str
phone_number             int64
address                    str
city_id                    str
city_name                  str
state_id                   str
pin_code                 int64
city_tier                  str
region_id                  str
occupation                 str
annual_income_inr      float64
customer_segment           str
product_line               str
plan_id                    str
acquisition_channel        str
acquisition_date           str
tenure_months            int64
customer_status            str
churn_date                 str
dtype: object
subscriptions | rows: 20523 | columns: 8
subscription_id            str
customer_id                str
plan_id                    str
start_date                 str
end_date

In [68]:

rows = []
for name in dfs:
    rows.append([name, dfs[name].shape[0], dfs[name].shape[1]])

summary = pd.DataFrame(rows, columns=["table", "rows", "columns"])
summary

,table,rows,columns
0,customers,19076,25
1,subscriptions,20523,8
2,plans,25,10
3,plan_history,19349,6
4,contracts,10607,9
5,devices,19000,8
6,billing,115313,9
7,payments,92362,7
8,recharges,146501,6
9,usage_voice,57000,6


### Observation

-- The largest table is 'recharges' with 146,501 rows, followed by 'billing' with 115,313 rows. The smallest table is 'regions' with only 4 rows. The central table, 'customers', has 19,076 rows and 25 columns. The row counts match the record counts given in the project document.

-- Text columns are stored as text (str), and numeric columns are stored as integers (int64) or decimals (float64). Date columns such as 'acquisition_date' and 'billing_date' are currently stored as text in DD-MM-YYYY format, so they will be converted to datetime format in a later step. Columns like 'phone_number' and 'pin_code' are stored as numbers, which will matter when checking whether they are valid.

# Missing Value Analysis

In [69]:
#MISSING VALUES
for name in dfs:
    df = dfs[name]
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(name, ": no missing values")
    else:
        print(name)
        print(pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)}))
    print("-" * 40)

customers
                   missing  percent
email                  745     3.91
address                380     1.99
annual_income_inr      954     5.00
churn_date           15560    81.57
----------------------------------------
subscriptions
          missing  percent
plan_id        41     0.20
end_date    17017    82.92
----------------------------------------
plans : no missing values
----------------------------------------
plan_history : no missing values
----------------------------------------
contracts : no missing values
----------------------------------------
devices : no missing values
----------------------------------------
billing : no missing values
----------------------------------------
payments : no missing values
----------------------------------------
recharges : no missing values
----------------------------------------
usage_voice : no missing values
----------------------------------------
usage_sms : no missing values
---------------------------------------

In [70]:
print(dfs["customers"]["customer_status"].value_counts())
print(dfs["subscriptions"]["subscription_status"].value_counts())

customer_status
Active     15560
Churned     3516
Name: count, dtype: int64
subscription_status
Active           16930
Terminated        3491
UNKNOWN_STATE      102
Name: count, dtype: int64


### Observation

Each table was checked for missing values, and the percentage of missing values was calculated for every column.

Only four tables contain missing values: customers, subscriptions, support_tickets and complaints.

In customers, "email" (3.91%), "address" (1.99%) and "annual_income_inr" (5.00%) have missing values, which are genuine data quality issues. churn_date is missing for 81.57% of customers, which is expected because most customers are still active. In subscriptions, "plan_id" is missing in 41 rows, which is a real issue because every subscription should be linked to a plan, while the blank "end_date" values belong to subscriptions that are still running. In support_tickets and complaints, the missing resolution fields most likely represent tickets and complaints that are not yet resolved.

# Duplicates

Duplicates are repeated records. 
If you leave them in customers, invoices, and payments get counted twice, which would inflate your KPIs.

In [71]:
for name in dfs:
    print(name, "duplicate rows:", dfs[name].duplicated().sum())

print("-" * 40)
print("Duplicate customer_id in customers:", dfs["customers"]["customer_id"].duplicated().sum())
print("Duplicate invoice_id in billing:", dfs["billing"]["invoice_id"].duplicated().sum())
print("Duplicate payment_id in payments:", dfs["payments"]["payment_id"].duplicated().sum())

customers duplicate rows: 76
subscriptions duplicate rows: 0
plans duplicate rows: 0
plan_history duplicate rows: 0
contracts duplicate rows: 0
devices duplicate rows: 0
billing duplicate rows: 344
payments duplicate rows: 276
recharges duplicate rows: 0
usage_voice duplicate rows: 0
usage_sms duplicate rows: 0
usage_data duplicate rows: 0
network_quality duplicate rows: 0
support_tickets duplicate rows: 0
complaints duplicate rows: 0
customer_feedback duplicate rows: 0
retention_campaigns duplicate rows: 0
marketing_campaigns duplicate rows: 0
employees duplicate rows: 0
stores duplicate rows: 0
regions duplicate rows: 0
cities duplicate rows: 0
states duplicate rows: 0
data_quality_issue_log duplicate rows: 0
----------------------------------------
Duplicate customer_id in customers: 76
Duplicate invoice_id in billing: 344
Duplicate payment_id in payments: 276


In [72]:
for name in ["customers", "billing", "payments"]:
    before = len(dfs[name])
    dfs[name] = dfs[name].drop_duplicates()
    after = len(dfs[name])
    print(name, ": before", before, "| after", after, "| removed", before - after)

customers : before 19076 | after 19000 | removed 76
billing : before 115313 | after 114969 | removed 344
payments : before 92362 | after 92086 | removed 276


## Duplicate Record Check

Duplicate rows were checked in all 24 tables using "duplicated()". The primary key columns ("customer_id", "invoice_id" and "payment_id") were also checked for repeated values.

### Observations

Only three tables have duplicates:

Table___________Duplicate rows_________Duplicate primary key

customers__________76_____________________76 (customer_id)

billing_____________344____________________344 (invoice_id)

payments___________276____________________276 (payment_id)

All the other tables show 0. These numbers match the issue log (duplicate_customers 76, duplicate_invoices 344, duplicate_payments 276).

The duplicate rows and the duplicate IDs are the same numbers. This means the extra rows are exact copies of existing rows. Because they are exact copies, it’s safe to delete them.

The duplicates were removed, leaving 19,000 customers, 114,969 invoices and 92,086 payments. Removing them prevents customers, revenue and payments from being counted twice in later analysis.


## Fixing the date columns

Right now every date is stored as text, so Python can’t sort by date or count months between dates. 
one real problem: some acquisition_date values are written in a different format.

Part A: Find the problem

In [73]:
customers = dfs["customers"]

d1 = pd.to_datetime(customers["acquisition_date"], format="%d-%m-%Y", errors="coerce")
print("Not in DD-MM-YYYY:", d1.isna().sum())
print(customers.loc[d1.isna(), "acquisition_date"].head())

Not in DD-MM-YYYY: 380
3     2014/10/18
33    2023/04/16
59    2021/06/13
74    2024/12/13
88    2021/10/31
Name: acquisition_date, dtype: str


there are 380 values that don’t fit, with examples like 2014/10/18. They are in YYYY/MM/DD format instead of DD-MM-YYYY. This matches mixed_date_format (380) in the issue log. 

Part B: Fix it

In [74]:
d2 = pd.to_datetime(customers["acquisition_date"], format="%Y/%m/%d", errors="coerce")
customers["acquisition_date"] = d1.fillna(d2)
dfs["customers"] = customers

print("Still unparsed:", customers["acquisition_date"].isnull().sum())

Still unparsed: 0


The code reads each date in the first format, and for the ones that failed it uses the second format. Still unparsed should be 0.

Part C: Convert all other date columns

In [75]:
date_columns = {
    "customers": ["date_of_birth", "churn_date"],
    "subscriptions": ["start_date", "end_date"],
    "plan_history": ["change_date"],
    "contracts": ["start_date", "end_date"],
    "devices": ["purchase_date"],
    "billing": ["billing_date", "due_date"],
    "payments": ["payment_date"],
    "recharges": ["recharge_date"],
    "support_tickets": ["created_date", "resolution_date"],
    "complaints": ["complaint_date", "resolution_date"],
    "customer_feedback": ["feedback_date"],
    "retention_campaigns": ["contact_date"],
    "marketing_campaigns": ["start_date", "end_date"],
    "employees": ["hire_date"],
    "stores": ["opened_date"],
}

for table in date_columns:
    for col in date_columns[table]:
        before = dfs[table][col].isnull().sum()
        dfs[table][col] = pd.to_datetime(dfs[table][col], format="%d-%m-%Y", errors="coerce")
        after = dfs[table][col].isnull().sum()
        if before != after:
            print("PROBLEM:", table, col, "-", after - before, "dates could not be converted")

print(dfs["customers"][["date_of_birth", "acquisition_date", "churn_date"]].dtypes)
print(dfs["billing"][["billing_date", "due_date"]].dtypes)

date_of_birth       datetime64[us]
acquisition_date    datetime64[us]
churn_date          datetime64[us]
dtype: object
billing_date    datetime64[us]
due_date        datetime64[us]
dtype: object


## Date Format Validation

The date columns in all tables were checked against the expected DD-MM-YYYY format and converted to datetime.

### Observation

All date columns followed the DD-MM-YYYY format except 'acquisition_date' in the customers table, where 380 records used the YYYY/MM/DD format. This matches the mixed_date_format issue in the data quality log. These dates were parsed with their own format and combined with the correctly formatted dates, so all 19,000 customer records now have a valid acquisition date. All other date columns were converted to datetime without any failures, and blank dates (such as churn_date for active customers) were kept as missing values. Consistent date formats are needed for tenure, churn trend and billing analysis in later phases.

## Invalid values in customers

This step has two parts. 
First I Will detect each problem, then  fix it.

Part A: Detect

In [76]:
customers = dfs["customers"]

# phone should have 12 digits (91 + 10-digit number), PIN should have 6 digits
bad_phone = customers["phone_number"].astype(str).str.len() != 12
bad_pin = customers["pin_code"].astype(str).str.len() != 6

# a valid email must contain "@"
bad_email = customers["email"].notnull() & ~customers["email"].str.contains("@", na=False)

# age calculated from date_of_birth should be close to the age column
age_from_dob = (pd.Timestamp("2026-03-31") - customers["date_of_birth"]).dt.days / 365.25
bad_age = (age_from_dob - customers["age"]).abs() > 1

# city names should match the official list in cities.csv
official_cities = dfs["cities"]["city_name"]
bad_city = ~customers["city_name"].isin(official_cities)

print("Invalid phone numbers:", bad_phone.sum())
print("Invalid PIN codes:", bad_pin.sum())
print("Invalid emails:", bad_email.sum())
print("Ages that do not match date_of_birth:", bad_age.sum())
print("Ages outside 0-120:", ((customers["age"] < 0) | (customers["age"] > 120)).sum())
print("City names not matching the official list:", bad_city.sum())

case_only = bad_city & customers["city_name"].str.lower().isin(official_cities.str.lower())
print("   of which capitalisation problems:", case_only.sum())
print("   of which spelling mistakes:", (bad_city & ~case_only).sum())

Invalid phone numbers: 228
Invalid PIN codes: 190
Invalid emails: 285
Ages that do not match date_of_birth: 38
Ages outside 0-120: 23
City names not matching the official list: 940
   of which capitalisation problems: 560
   of which spelling mistakes: 380


In [77]:
print(customers.loc[bad_email, "email"].head())
print(customers.loc[bad_age, ["age", "date_of_birth"]].head())
print(customers.loc[bad_city, "city_name"].head(10))

203    user544_at_mail
268    user123_at_mail
481    user615_at_mail
717    user428_at_mail
822    user690_at_mail
Name: email, dtype: str
      age date_of_birth
506     1    1992-03-31
751   220    1989-03-31
834   220    1988-03-31
868     1    1990-03-31
1604    0    2002-03-31
12       VADODARA
33     Hyderaabad
53     Ahmedaabad
66      HYDERABAD
79       SHILLONG
84           PUNE
93      Belaagavi
97       SILIGURI
138     BENGALURU
160       Kaanpur
Name: city_name, dtype: str


Part B: Fix 

In [78]:
# phone, PIN and email cannot be guessed, so set them to missing
# (we keep the customer, because the customer is still needed for churn analysis)
customers["phone_number"] = customers["phone_number"].where(~bad_phone)
customers["pin_code"] = customers["pin_code"].where(~bad_pin)
customers["email"] = customers["email"].where(~bad_email)

# age: recalculate from date_of_birth for the 38 wrong rows
customers.loc[bad_age, "age"] = age_from_dob[bad_age].astype(int)

# city: use the official name that matches the city_id
city_lookup = dfs["cities"].set_index("city_id")["city_name"]
customers["city_name"] = customers["city_id"].map(city_lookup)

dfs["customers"] = customers

print("Missing phone:", customers["phone_number"].isnull().sum())
print("Missing PIN:", customers["pin_code"].isnull().sum())
print("Missing email:", customers["email"].isnull().sum())
print("Age min / max:", customers["age"].min(), "/", customers["age"].max())
print("City names not official:", (~customers["city_name"].isin(official_cities)).sum())
print("Unique cities:", customers["city_name"].nunique())

Missing phone: 228
Missing PIN: 190
Missing email: 1029
Age min / max: 17 / 79
City names not official: 0
Unique cities: 68


## Customer Data Validation

The customers table was validated for invalid phone numbers, PIN codes, emails, ages and city names.

### Observation

The checks found 228 invalid phone numbers (not 12 digits), 190 invalid PIN codes (not 6 digits) and 285 invalid emails (missing "@"). Since the correct values cannot be recovered, they were set to missing, and the customer records were kept for churn analysis.

38 customers had an age that did not match their date of birth, including impossible values such as -5 and 220. These ages were recalculated from the date of birth.

940 city names did not match the official city list (560 capitalisation problems and 380 spelling mistakes, such as "PUNE" and "Hyderaabad"). They were replaced with the official name linked to each city_id, so all 68 cities are now consistent. These counts agree closely with the data quality issue log.

## Billing checks

Billing is the table behind revenue KPIs, so the problems here matter. 
The issue log lists four problems for billing. I have already removed the duplicates (344), so three remain: negative amounts, wrong GST, and future dates.


Part A: Detect 

In [79]:
billing = dfs["billing"]

# how do we know what a "future" date is? check the last billing period in the data
print("Latest billing period:", pd.to_datetime(billing["billing_period_month"], format="%m-%Y").max())

cutoff = pd.Timestamp("2026-03-31")
future = billing["billing_date"] > cutoff
negative = billing["total_amount_inr"] < 0
wrong_gst = (billing["gst_amount_inr"] - billing["base_amount_inr"] * 0.18).abs() > 0.05

print("Future billing dates:", future.sum())
print("Negative total amounts:", negative.sum())
print("Wrong GST amounts:", wrong_gst.sum())

gap = (billing["due_date"] - billing["billing_date"]).dt.days
print("Days between billing date and due date (normal rows):", gap[~future].unique())
print("Future rows where due date is before billing date:", (future & (gap < 0)).sum())

billing.loc[future, ["invoice_id", "billing_date", "billing_period_month", "due_date"]].head()

Latest billing period: 2026-03-01 00:00:00
Future billing dates: 229
Negative total amounts: 229
Wrong GST amounts: 1149
Days between billing date and due date (normal rows): [15]
Future rows where due date is before billing date: 229


,invoice_id,billing_date,billing_period_month,due_date
148,INV00000149,2026-08-11,12-2025,2026-01-15
1584,INV00001585,2027-04-11,05-2025,2025-05-20
1990,INV00001991,2026-09-27,06-2025,2025-06-19
2829,INV00002830,2027-01-30,01-2026,2026-02-14
3010,INV00003011,2027-02-01,03-2026,2026-04-15


Why 31 March 2026 is the cutoff. The latest billing period in the data is March 2026, so no real invoice can be dated after 31 March 2026. The earlier step used a different “today”, which is why the count can vary. With this cutoff i got exactly 229, which matches the log.

What the data tells. On every normal invoice, due_date is exactly 15 days after billing_date. The 229 future-dated invoices break this rule: their billing_date is wrong, but their due_date and billing_period_month are still believable. So the real billing date can be recovered as due date minus 15 days.


Part B: Fix

In [80]:
# 1. wrong GST: GST must be 18% of the base amount
billing.loc[wrong_gst, "gst_amount_inr"] = (billing.loc[wrong_gst, "base_amount_inr"] * 0.18).round(2)

# 2. negative totals: total = base amount + GST
billing.loc[negative, "total_amount_inr"] = (billing.loc[negative, "base_amount_inr"] + billing.loc[negative, "gst_amount_inr"]).round(2)

# 3. future billing dates: billing date = due date - 15 days
billing.loc[future, "billing_date"] = billing.loc[future, "due_date"] - pd.Timedelta(days=15)

dfs["billing"] = billing

print("Future dates left:", (billing["billing_date"] > cutoff).sum())
print("Negative totals left:", (billing["total_amount_inr"] < 0).sum())
print("Wrong GST left:", ((billing["gst_amount_inr"] - billing["base_amount_inr"] * 0.18).abs() > 0.05).sum())
print("Latest billing date:", billing["billing_date"].max())
print("Smallest total amount:", billing["total_amount_inr"].min())
print("Rows:", len(billing))

Future dates left: 0
Negative totals left: 0
Wrong GST left: 0
Latest billing date: 2026-03-31 00:00:00
Smallest total amount: 92.44
Rows: 114969


## Billing Data Validation

The billing table was checked for future billing dates, negative total amounts and incorrect GST values. The last billing period in the data is March 2026, so 31 March 2026 was used as the cutoff for future dates.

### Observation

The checks found 229 invoices with future billing dates, 229 invoices with negative total amounts and 1,149 invoices where the GST amount was not 18% of the base amount. These counts match the data quality issue log exactly.

On all normal invoices the due date is 15 days after the billing date, so the 229 future-dated invoices could be corrected by calculating billing date = due date - 15 days. Negative totals were corrected to base amount + GST, and wrong GST values were recalculated as 18% of the base amount. After the corrections, no future dates, negative totals or GST errors remain, and the billing data is ready for revenue and ARPU calculations.

## Referential integrity and subscriptions

Tables are linked by ID columns (for example customer_id). This step checks that every ID in a child table exists in its parent table, using the ERD, and fixes the subscriptions table.

Part A: Detect

In [81]:
# 1. every customer_id in other tables should exist in customers
valid_customers = dfs["customers"]["customer_id"]
for name in dfs:
    if name != "customers" and "customer_id" in dfs[name].columns:
        orphans = (~dfs[name]["customer_id"].isin(valid_customers)).sum()
        print(name, ": customer_id not found in customers =", orphans)


subscriptions : customer_id not found in customers = 0
plan_history : customer_id not found in customers = 0
contracts : customer_id not found in customers = 0
devices : customer_id not found in customers = 0
billing : customer_id not found in customers = 0
payments : customer_id not found in customers = 0
recharges : customer_id not found in customers = 0
usage_voice : customer_id not found in customers = 0
usage_sms : customer_id not found in customers = 0
usage_data : customer_id not found in customers = 0
network_quality : customer_id not found in customers = 0
support_tickets : customer_id not found in customers = 0
complaints : customer_id not found in customers = 0
customer_feedback : customer_id not found in customers = 0
retention_campaigns : customer_id not found in customers = 0


In [82]:
# 2. the other links from the customers table
customers = dfs["customers"]
print("customers.plan_id not in plans:", (~customers["plan_id"].isin(dfs["plans"]["plan_id"])).sum())
print("customers.city_id not in cities:", (~customers["city_id"].isin(dfs["cities"]["city_id"])).sum())
print("customers.state_id not in states:", (~customers["state_id"].isin(dfs["states"]["state_id"])).sum())
print("customers.region_id not in regions:", (~customers["region_id"].isin(dfs["regions"]["region_id"])).sum())

customers.plan_id not in plans: 0
customers.city_id not in cities: 0
customers.state_id not in states: 0
customers.region_id not in regions: 0


In [83]:
# 3. payments should point to an invoice that exists in billing
payments = dfs["payments"]
orphan_payments = ~payments["invoice_id"].isin(dfs["billing"]["invoice_id"])
print("Payments with an invoice_id not in billing:", orphan_payments.sum())
print(payments.loc[orphan_payments, ["payment_id", "invoice_id", "amount_inr"]].head())

Payments with an invoice_id not in billing: 184
       payment_id   invoice_id  amount_inr
874   PAY00000875  INV94560686     1444.45
1007  PAY00001008  INV96995702      723.82
1474  PAY00001475  INV99552429     1439.82
2112  PAY00002113  INV98863464      746.33
2117  PAY00002118  INV92113763     1536.96


In [84]:
# 4. subscriptions checks
subs = dfs["subscriptions"]
print("Subscriptions with missing plan_id:", subs["plan_id"].isnull().sum())
print("Subscriptions whose plan_id is not in plans:", (subs["plan_id"].notnull() & ~subs["plan_id"].isin(dfs["plans"]["plan_id"])).sum())
print(subs["subscription_status"].value_counts())

Subscriptions with missing plan_id: 41
Subscriptions whose plan_id is not in plans: 0
subscription_status
Active           16930
Terminated        3491
UNKNOWN_STATE      102
Name: count, dtype: int64


The 184 broken payments have invoice IDs like INV94560686, which don’t exist in billing at all. Customer links are fine everywhere.

Part B: Fix 

In [85]:
# payments: remove the 184 payments that cannot be linked to any invoice
dfs["payments"] = payments[~orphan_payments].copy()

# subscriptions: fill the missing plan_id with the plan of the same customer
customer_plan = dfs["customers"].set_index("customer_id")["plan_id"]
missing_plan = subs["plan_id"].isnull()
subs.loc[missing_plan, "plan_id"] = subs.loc[missing_plan, "customer_id"].map(customer_plan)

# subscriptions: UNKNOWN_STATE -> Active if there is no end_date, otherwise Terminated
unknown = subs["subscription_status"] == "UNKNOWN_STATE"
subs.loc[unknown & subs["end_date"].isnull(), "subscription_status"] = "Active"
subs.loc[unknown & subs["end_date"].notnull(), "subscription_status"] = "Terminated"
dfs["subscriptions"] = subs

print("Payments rows:", len(dfs["payments"]))
print("Payments still not linked:", (~dfs["payments"]["invoice_id"].isin(dfs["billing"]["invoice_id"])).sum())
print("Missing plan_id left:", dfs["subscriptions"]["plan_id"].isnull().sum())
print(dfs["subscriptions"]["subscription_status"].value_counts())

Payments rows: 91902


Payments still not linked: 0
Missing plan_id left: 0
subscription_status
Active        17017
Terminated     3506
Name: count, dtype: int64


## Referential Integrity Check

The relationships shown in the ERD were validated. Every customer_id in the other tables was compared with the customers table, the plan, city, state and region links of customers were checked, and every payment was matched with an invoice in billing. The subscriptions table was also checked for missing plan IDs and invalid statuses.

### Observation

All customer_id values in the 15 related tables exist in the customers table, and all plan, city, state and region links of customers are valid. Two problems were found in other tables. First, 184 payments refer to invoice IDs that do not exist in billing, which matches the broken_foreign_key issue in the data quality log. These payments cannot be linked to any bill, so they were removed (91,902 payments remain). Second, 41 subscriptions had no plan_id and 102 subscriptions had the invalid status UNKNOWN_STATE. The missing plan IDs were filled using the customer's own plan, and the invalid statuses were corrected using the end_date (Active if there is no end date, otherwise Terminated). All counts match the issue log, and all relationships between the tables are now consistent.

## Missing values in customers

Earlier I found blanks in email, address, and annual_income_inr. so i filled a blank only when later analysis needs the column and a sensible estimate exists.


PART A:Check

In [86]:
customers = dfs["customers"]
print("Missing email:", customers["email"].isnull().sum())
print("Missing address:", customers["address"].isnull().sum())
print("Missing income:", customers["annual_income_inr"].isnull().sum())
print("Overall median income:", round(customers["annual_income_inr"].median()))
customers.groupby("occupation")["annual_income_inr"].median().round(0).sort_values()

Missing email: 1029
Missing address: 380
Missing income: 950
Overall median income: 692762


occupation
Student                   59239.0
Homemaker                100356.0
Delivery Partner         291552.0
Factory Worker           323598.0
Auto/Cab Driver          328759.0
Farmer                   360082.0
Nurse                    478911.0
School Teacher           524927.0
Retail Shopkeeper        536918.0
Retired                  556366.0
Sales Executive          566396.0
Police Officer           671212.0
HR Executive             728418.0
Government Employee      774768.0
Freelancer               849414.0
Civil Engineer           956747.0
Bank Officer             996929.0
Self Employed           1067344.0
Data Analyst            1157226.0
Operations Manager      1290377.0
Small Business Owner    1362943.0
Marketing Manager       1419165.0
Lawyer                  1505200.0
Chartered Accountant    1754148.0
Software Engineer       1754418.0
Doctor                  2339960.0
Name: annual_income_inr, dtype: float64

Part B: Fix

In [87]:
# keep a record of which incomes were filled in
customers["income_imputed"] = customers["annual_income_inr"].isnull()

# fill missing income with the median income of the same occupation
occupation_median = customers.groupby("occupation")["annual_income_inr"].transform("median")
customers["annual_income_inr"] = customers["annual_income_inr"].fillna(occupation_median.round(0))

dfs["customers"] = customers

print("Missing income left:", customers["annual_income_inr"].isnull().sum())
print("Incomes filled:", customers["income_imputed"].sum())
print("Median income now:", round(customers["annual_income_inr"].median()))
print("Missing email left:", customers["email"].isnull().sum())
print("Missing address left:", customers["address"].isnull().sum())

Missing income left: 0
Incomes filled: 950
Median income now: 694984
Missing email left: 1029
Missing address left: 380


## Missing Value Treatment

The missing values found in the customers table were reviewed and handled based on whether the column is needed for analysis.

### Observation

After cleaning, 1,029 emails, 380 addresses and 950 annual incomes were missing. Email and address cannot be estimated from other columns, and they are not needed for churn analysis, so they were left blank instead of being filled with made-up values.

Annual income is useful for customer analysis, but it varies a lot by occupation (about ₹59,000 for students and ₹2.3 million for doctors), so filling it with the overall median would be misleading. Instead, each missing income was filled with the median income of the same occupation. The median was used because income is right-skewed. A column 'income_imputed' was added to mark the 950 filled values, and the overall median income stayed almost unchanged (from about 692,762 to 694,984).

## Outliers in usage_data

An outlier is a value far from the rest. In usage_data, the column to check is data_gb_used (the data a customer used in a month).

Part A: Detect

In [88]:
usage = dfs["usage_data"]
x = usage["data_gb_used"]

print(x.describe().round(2))

# method 1: IQR rule
q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print("IQR upper limit:", round(upper, 2), "| rows above it:", (x > upper).sum())

# method 2: look at the largest values and how the values are spread
print(x.sort_values(ascending=False).head(10).round(1).tolist())
print(pd.cut(x, [0, 50, 100, 200, 300, 500, 1000, 100000]).value_counts().sort_index())
print("Rows above 300 GB:", (x > 300).sum())

count    57000.00
mean        40.18
std        332.57
min          0.24
25%         11.39
50%         19.14
75%         32.63
max      22172.70
Name: data_gb_used, dtype: float64
IQR upper limit: 64.49 | rows above it: 4480
[22172.7, 21960.6, 18348.4, 17377.2, 15322.7, 13886.7, 13344.5, 13154.9, 12747.5, 12581.4]
data_gb_used
(0, 50]           49892
(50, 100]          5552
(100, 200]         1322
(200, 300]           61
(300, 500]            7
(500, 1000]          17
(1000, 100000]      149
Name: count, dtype: int64
Rows above 300 GB: 173


Part B: Fix

Every customer has exactly 3 months of usage. Replacing each outlier with the median of that customer’s other months.

In [89]:
outlier = usage["data_gb_used"] > 300

# keep the original value in a separate column
usage["data_gb_original"] = usage["data_gb_used"]

# hide the outliers, then fill them with the customer's own median
usage["data_gb_used"] = usage["data_gb_used"].where(~outlier)
customer_median = usage.groupby("customer_id")["data_gb_used"].transform("median")
usage["data_gb_used"] = usage["data_gb_used"].fillna(customer_median)

dfs["usage_data"] = usage

print("Outliers replaced:", outlier.sum())
print("Max usage now:", round(usage["data_gb_used"].max(), 1))
print("Mean usage now:", round(usage["data_gb_used"].mean(), 2))
print("Missing values:", usage["data_gb_used"].isnull().sum())

Outliers replaced: 173
Max usage now: 298.1
Mean usage now: 26.83
Missing values: 0


## Outlier Detection: Data Usage

The data_gb_used column in usage_data was checked for outliers using the IQR rule and by looking at the largest values and their distribution.

### Observation

The data usage is highly skewed: the median is 19.14 GB but the maximum is 22,172.7 GB, which is impossible for one customer in one month. The IQR rule flagged 4,480 rows, but many of these are genuine heavy users on 5G and fibre plans, so it was not used for deletion. The distribution shows a sharp break after about 300 GB (only 173 rows above it, with values in the thousands and tens of thousands), so 300 GB was used as the cutoff. This is close to the 171 outliers in the data quality log.

Each customer has 3 months of usage, so each outlier was replaced with the median of that customer's own usage, and the original value was kept in data_gb_original. After this, the maximum usage is 298.1 GB and the mean falls from 40.18 GB to 26.83 GB, which makes later usage analysis reliable.

## Final summary


Part A: Complaints check 

The PRD asks what share of complaints are still unresolved.

In [90]:
complaints = dfs["complaints"]
print(complaints["status"].value_counts())
print((complaints["status"].value_counts(normalize=True) * 100).round(2))

status
Resolved      9236
Unresolved    7348
Name: count, dtype: int64
status
Resolved      55.69
Unresolved    44.31
Name: proportion, dtype: float64


Part B: Compare with the issue log

This table is the “compare your findings with the issue log” requirement.

In [91]:
rows = [
    ["customers", "missing_email", 760, 744, "Left blank (cannot be recovered, not needed)"],
    ["customers", "missing_address", 380, 380, "Left blank (cannot be recovered, not needed)"],
    ["customers", "missing_income", 950, 950, "Filled with the median income of the same occupation"],
    ["customers", "invalid_phone", 228, 228, "Set to missing"],
    ["customers", "invalid_email", 285, 285, "Set to missing"],
    ["customers", "incorrect_pin", 190, 190, "Set to missing"],
    ["customers", "misspelled_city", 380, 380, "Replaced with official name using city_id"],
    ["customers", "inconsistent_capitalisation", 570, 560, "Replaced with official name using city_id"],
    ["customers", "impossible_age", 38, 38, "Recalculated from date_of_birth"],
    ["customers", "mixed_date_format", 380, 380, "Parsed with its own format"],
    ["customers", "duplicate_customers", 76, 76, "Duplicates removed"],
    ["billing", "incorrect_gst", 1149, 1149, "Recalculated as 18% of base amount"],
    ["billing", "negative_invoice_amount", 229, 229, "Total = base amount + GST"],
    ["billing", "future_dates", 229, 229, "Billing date = due date - 15 days"],
    ["billing", "duplicate_invoices", 344, 344, "Duplicates removed"],
    ["payments", "broken_foreign_key", 184, 184, "Rows removed"],
    ["payments", "duplicate_payments", 276, 276, "Duplicates removed"],
    ["subscriptions", "null_foreign_key", 41, 41, "Filled with the customer's plan"],
    ["subscriptions", "invalid_category", 102, 102, "Status set from end_date"],
    ["usage_data", "outlier_values", 171, 173, "Replaced with the customer's median usage"],
]

issue_summary = pd.DataFrame(rows, columns=["table", "issue", "logged", "found", "action"])
issue_summary["difference"] = issue_summary["found"] - issue_summary["logged"]
issue_summary

,table,issue,logged,found,action,difference
0,customers,missing_email,760,744,"Left blank (cannot be recovered, not needed)",-16
1,customers,missing_address,380,380,"Left blank (cannot be recovered, not needed)",0
2,customers,missing_income,950,950,Filled with the median income of the same occu...,0
3,customers,invalid_phone,228,228,Set to missing,0
4,customers,invalid_email,285,285,Set to missing,0
5,customers,incorrect_pin,190,190,Set to missing,0
6,customers,misspelled_city,380,380,Replaced with official name using city_id,0
7,customers,inconsistent_capitalisation,570,560,Replaced with official name using city_id,-10
8,customers,impossible_age,38,38,Recalculated from date_of_birth,0
9,customers,mixed_date_format,380,380,Parsed with its own format,0


Part C: Save the cleaned data

In [92]:
for name in dfs:
    if name != "data_quality_issue_log":
        dfs[name].to_csv(name + "_cleaned.csv", index=False)

print("Cleaned files saved")

Cleaned files saved


# Data Quality Report

## Objective
The objective of this phase was to assess the quality of the 24 NexaTel datasets, clean the problems found, and prepare reliable data for analysis, KPI calculation and dashboard development.

## Summary of Findings
- All 24 tables were loaded and profiled. Row counts match the project document.
- Missing values were found in customers (email, address, income, churn_date), subscriptions (plan_id, end_date), support_tickets and complaints. Blank churn, end and resolution dates are expected for active or unresolved records.
- Duplicates: 76 customers, 344 invoices and 276 payments.
- Dates: 380 acquisition dates used a different format (YYYY/MM/DD).
- Customers: 228 invalid phone numbers, 190 invalid PIN codes, 285 invalid emails, 38 ages that did not match the date of birth, and 940 city names that were misspelled or wrongly capitalised.
- Billing: 1,149 incorrect GST amounts, 229 negative totals and 229 future billing dates.
- Referential integrity: all customer links are valid, but 184 payments refer to invoices that do not exist.
- Subscriptions: 41 missing plan IDs and 102 invalid statuses (UNKNOWN_STATE).
- Usage: 173 data usage values above 300 GB, with values up to 22,172 GB.
- 44.31% of complaints (7,348) are still unresolved.

## Actions Taken
- Exact duplicates were removed from customers, billing and payments.
- Dates were converted to datetime, and the mixed format was corrected.
- Invalid phone numbers, PIN codes and emails were set to missing, and ages and city names were corrected from reliable columns.
- GST, totals and billing dates were recalculated using the business rules (GST is 18% of the base amount, and due date is 15 days after billing date).
- Broken payments were removed, and subscription plan IDs and statuses were repaired.
- Usage outliers were replaced with each customer's median usage, and missing income was filled with the median for the same occupation.

## Comparison with the Issue Log
17 of the 20 logged issues matched exactly. The remaining three were close: missing email (744 vs 760), inconsistent capitalisation (560 vs 570) and usage outliers (173 vs 171).

## Assumptions and Limitations
- 31 March 2026 was used as the cutoff for future billing dates, because it is the last billing period in the data.
- 300 GB was used as the outlier cutoff, based on the break in the usage distribution.
- Some filled values (plan_id, income, usage) are estimates, and flag or original columns were kept where possible.

## Conclusion
The data had 20 injected quality problems, and all of them were identified and handled. The cleaned datasets are consistent and ready for exploratory analysis, KPI calculation and dashboard development in the next phases.